In [1]:
import json
import random

# 文件路径
file_path = '/home/rickc/Projects/LLMs/data/SFT/merged_output.json'

# 打开并读取JSON文件
with open(file_path, 'r', encoding='utf-8') as file:
    # 加载JSON数据到一个Python对象（这里假设是列表）
    dataset = json.load(file)


 # 随机选择不同的条目
random_entries = random.sample(dataset, 2)

# 打印随机选择的条目
for index, entry in enumerate(random_entries, start=1):
    print(f"Entry {index}:")
    print(json.dumps(entry, indent=4, ensure_ascii=False))
    print('-' * 40)  # 分隔线

Entry 1:
{
    "instruction": "给我讲一个笑话",
    "input": "",
    "output": "科技公司老总王老汉钟爱柯基犬只因其有柯基感, 科技公司老总王老汉钟爱柯基犬只因其有柯基感",
    "system": "",
    "history": []
}
----------------------------------------
Entry 2:
{
    "instruction": "如何让中年男同事1秒回春，作者阿猛的秘诀是什么？",
    "input": "",
    "output": "我看到的标题是：\n做对这几步，中年男同…\n这省略号真是省在了对的地方[辣眼睛][辣眼睛][辣眼睛]",
    "system": "You are a helpful assistant",
    "history": []
}
----------------------------------------


In [ ]:
# #验证SDK token
# from modelscope.hub.api import HubApi
# api = HubApi()
# api.login('***')

In [ ]:
# !modelscope download --dataset 'arickc1999/SpeechLM/' merged_output.json --cache_dir './data'

In [2]:
# 定义全局变量和参数
model_name_or_path = '/home/rickc/Projects/LLMs/models/Qwen/Qwen2.5-0.5B'  # 模型ID或本地路径
# train_data_path = 'data/zhouyi_dataset_handmade.csv'    # 训练数据路径
train_data_path = '/home/rickc/Projects/LLMs/data/SFT/merged_output.json'    # 训练数据路径(批量生成数据集）
eval_data_path = None                     # 验证数据路径，如果没有则设置为None
max_input_length = 128                    # 输入的最大长度
max_output_length = 256                  # 输出的最大长度
prompt_text = ''                          # 所有数据前的指令文本
seed = 234                                 # 随机种子

In [3]:
def format_input(entry):
    instruction_text = (
        f"下面是一个描述任务的 instruction。"
        f"给出这个 instruction 恰当的回答。"
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [4]:
dataset[0]

{'instruction': '给我讲一个笑话',
 'input': '',
 'output': '狐假虎威就是爽，我提着领导的头出门，大家看到都吓跑了',
 'system': '',
 'history': []}

In [5]:
model_input = format_input(dataset[50])
desired_response = f"\n\n### Response:\n{dataset[50]['output']}"

print(model_input + desired_response)

下面是一个描述任务的 instruction。给出这个 instruction 恰当的回答。

### Instruction:
给我讲一个笑话

### Response:
一流院校组织学术流氓到双非大学踢馆


In [6]:
from datasets import Dataset

# 创建Dataset对象
dataset = Dataset.from_list(dataset)

# 分割数据集为训练集、验证集和测试集

# 首先对整个数据集进行打乱
shuffled_dataset = dataset.shuffle(seed=seed)

# 先将80%的数据分给训练集，剩下的20%用于后续分割
train_testvalid_split = shuffled_dataset.train_test_split(test_size=0.2, seed=seed)

# 再将剩余的20%数据分给验证集和测试集，各占50%
test_valid_split = train_testvalid_split['test'].train_test_split(test_size=0.5, seed=seed)

# 最终得到三个数据集
train_data = train_testvalid_split['train']
val_data = test_valid_split['train']  # 这里的'train'实际上是原始数据集的20%中的前半部分，也就是验证集
test_data = test_valid_split['test']    # 这里的'test'实际上是原始数据集的20%中的后半部分，也就是测试集

# 打印各数据集的大小以确认分割是否正确
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")


Training set size: 84805
Validation set size: 10601
Test set size: 10601


In [7]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [8]:
def custom_collate_fn(
    batch,
    pad_token_id=151643, # Qwen2.5_0.5B的pad_token_id
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# 下载模型
# from modelscope import snapshot_download

# snapshot_download('Qwen/Qwen2.5-0.5B', local_dir='/home/rickc/Projects/LLMs/models/Qwen/Qwen2.5-0.5B')

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = '/home/rickc/Projects/LLMs/models/Qwen/Qwen2.5-0.5B'  # 模型ID或本地路径

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [11]:
!nvidia-smi

Sun Dec 22 12:31:01 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.52.01              Driver Version: 555.99         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 ...    On  |   00000000:01:00.0  On |                  N/A |
| N/A   56C    P0             33W /  100W |    2022MiB /   8192MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
model.eval()

prompt = "大连老虎滩海洋公园是"

# text = tokenizer.encode(prompt)
model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=6
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [13]:
from functools import partial

# 固定 custom_collate_fn 的某些参数，从而创建一个新的函数 customized_collate_fn
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=256
)

In [14]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)
 
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [15]:
val_loader_test = tokenizer.decode(val_dataset[0])
val_loader_test

'下面是一个描述任务的 instruction。给出这个 instruction 恰当的回答。\n\n### Instruction:\n给我讲一个笑话\n\n### Response:\n披上羊皮的狼刚混进羊群没多久就碰上了苏格兰人, 披上羊皮的狼刚混进羊群没多久就碰上了苏格兰人'

In [16]:
# 获取第一个 batch 的第一个样本
first_batch = next(iter(test_loader))
first_sample_data, first_sample_target = first_batch[0][0], first_batch[1][0]

print("First sample data:", first_sample_data)
print("First sample target:", first_sample_target)

First sample data: tensor([100431, 101909,  53481,  88802,   9370,   7600,   1773, 107485,  99487,
          7600,   6567,    223,    108,  39165, 111423,   3407,  14374,  29051,
           510, 104169,  99526,  46944, 109959,    271,  14374,   5949,    510,
          2073, 105010,  29258,  29490,   3837, 104484, 111847,    854,  99950,
        117440,  32648, 108525, 103922,   3837, 101434,  44729,  30709,  99445,
         99948,  75606,  75606, 112173,  99561,   6313,     11,   1036, 105010,
         29258,  29490,   3837, 104484, 111847,    854,  99950, 117440,  32648,
        108525, 103922,   3837, 101434,  44729,  30709,  99445,  99948,  75606,
         75606, 112173,  99561,   6313, 151643, 151643, 151643, 151643, 151643,
        151643], device='cuda:0')
First sample target: tensor([101909,  53481,  88802,   9370,   7600,   1773, 107485,  99487,   7600,
          6567,    223,    108,  39165, 111423,   3407,  14374,  29051,    510,
        104169,  99526,  46944, 109959,    271

In [17]:
# 检查一下训练加载器生成的输入和目标批次的维度
print("Train loader:")
for batch_idx, (inputs, targets) in enumerate(train_loader):
    if batch_idx >= 10:
        break
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 102]) torch.Size([8, 102])
torch.Size([8, 85]) torch.Size([8, 85])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 135]) torch.Size([8, 135])
torch.Size([8, 81]) torch.Size([8, 81])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 233]) torch.Size([8, 233])
torch.Size([8, 125]) torch.Size([8, 125])
torch.Size([8, 91]) torch.Size([8, 91])
torch.Size([8, 78]) torch.Size([8, 78])


## 在指令数据上微调LLM

In [18]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())


In [19]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # 如果当前上下文超出支持的上下文大小，则裁剪当前上下文
        # 例如，如果 LLM 仅支持 5 个 token，而上下文大小为 10
        # 那么只有最后 5 个 token 用作上下文
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # 仅关注最后一个时间步骤
        # (batch, n_token, vocab_size) 变为 (batch, vocab_size)
        logits = logits.logits[:, -1, :]

        # 获取具有最高 logits 值的词汇条目的 idx
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # 将采样索引附加到正在运行的序列
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [20]:
from transformers import AutoConfig

# 加载 Qwen2 模型的配置
config = AutoConfig.from_pretrained(model_path)
qwen2_context_size = config.max_position_embeddings

In [21]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = qwen2_context_size
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [22]:
# Print a sample text after each epoch

start_context = "抚顺老虎台"
generate_and_print_sample(model, tokenizer, device, start_context)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


抚顺老虎台镇:打造"三张名片" 争当"全国文明村镇"  "老虎台镇是全国文明村镇,是全国农村精神文明建设示范镇,是全国农村基层治理示范镇,是全国农村基层党建示范镇。"


In [23]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [24]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [25]:
model.to(device)

torch.manual_seed(234)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 4.445145511627198
Validation loss: 4.36677885055542


In [ ]:
import time

start_time = time.time()

torch.manual_seed(234)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 3.176, Val loss 2.935
Ep 1 (Step 000005): Train loss 2.356, Val loss 2.323
Ep 1 (Step 000010): Train loss 2.469, Val loss 2.291
Ep 1 (Step 000015): Train loss 2.204, Val loss 2.268


KeyboardInterrupt: 